# TPWRS PRP Publications by Year

How the IEEE *Transactions on Power Systems* corpus tracks the 2025 Research Agenda, 2015-2026.

Reading notes that apply to every figure below:

- **Every trend is a share of that year's papers, never a count.** TPWRS volume swings from 358 papers
  (2015) to 703 (2018), so raw counts mostly measure how much the journal published.
- **2026 is a partial year** (collected to August 2026) and is shaded in every time series.
- **These are *coverage* measures**: each paper contributes its single best-matching question inside a
  programme, so a figure reads "how many papers this programme reaches", not "how much work each paper
  does". Pooling all abstract-question pairs instead would measure intensity and would be diluted by
  programmes holding many narrow questions.
- Level bands come from `data/prompts/affinity_scoring_scale.yaml`: Low <= 40, Moderate 40-80, High > 80.

## 1. Configuration

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.append(str(repo_root))
from src.analysis import assign_prps

BENCHMARK_ROOT = repo_root / "data" / "results" / "affinity_benchmark"
OUTPUT_DIR = repo_root / "notebooks" / "outputs" / "tpwrs_prp_publications_by_year"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROGRAMME_ORDER = ["IBR", "Stability", "DER", "CROF", "System Services", "Planning"]
LEVEL_ORDER = ["Low", "Moderate", "High"]
LEVEL_DTYPE = pd.CategoricalDtype(categories=LEVEL_ORDER, ordered=True)
THRESHOLDS = {"Moderate+": ["Moderate", "High"], "High": ["High"]}
ERA_BINS = [2014, 2017, 2020, 2023, 2026]
ERA_LABELS = ["2015-17", "2018-20", "2021-23", "2024-26"]
PARTIAL_YEARS = [2026]

# Palettes validated with the dataviz skill's scripts/validate_palette.py --mode light.
# The ordered Low/Moderate/High ramp and its 2-step subset pass --ordinal; the gain/loss pair passes
# --pairs all. The six-programme categorical set is NOT used for line charts: under --pairs all it
# fails (Planning #008300 vs Stability #eb6834 is dE 3.2 under protanopia), so programmes are separated
# by facet position here, not by hue.
LEVEL_COLORS = {"Low": "#86b6ef", "Moderate": "#2a78d6", "High": "#104281"}
BLUE_RAMP = ["#86b6ef", "#4b8fe0", "#2a78d6", "#104281"]
ACCENT, CONTEXT = "#104281", "#898781"
LOSS, GAIN = "#2a78d6", "#e34948"
SURFACE, INK, INK_SECONDARY, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
PARTIAL_BAND = "#f1f0ea"

latex_textwidth_pt = 516.0 # double column # use this command in latex: \the\textwidth
# latex_textwidth_pt = 452.0 # single column
scale = 1
FIGURE_DIM = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.6)

FOOTNOTE = (
    "*2026 partial (collected to Aug 2026)."
)
# pio.templates["gpst"] = go.layout.Template(
#     layout=dict(
#         paper_bgcolor=SURFACE,
#         plot_bgcolor=SURFACE,
#         font=dict(family="Helvetica Neue, Helvetica, Arial, sans-serif", size=12, color=INK_SECONDARY),
#         title=dict(font=dict(size=16, color=INK), x=0, xanchor="left", xref="paper", y=0.97, yanchor="top"),
#         xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor=GRID, linewidth=1,
#                    ticks="outside", ticklen=4, tickcolor=GRID, tickfont=dict(color=INK_MUTED, size=11)),
#         yaxis=dict(showgrid=True, gridcolor=GRID, gridwidth=1, zeroline=False, showline=False,
#                    ticks="", tickfont=dict(color=INK_MUTED, size=11)),
#         legend=dict(title_text="", orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
#                     font=dict(color=INK_SECONDARY, size=11)),
#         margin=dict(l=64, r=40, t=104, b=76),
#     )
# )
# pio.templates.default = "plotly_white"
# SAVE = True


## 2. Load the adjudicated PRP assignments

`assign_prps` returns one row per (paper, programme) holding that programme's best-matching RA2025
question, so every paper contributes exactly six rows and `Low + Moderate + High` sums to 100% within
any (year, programme) cell.

In [2]:
assignments_df = pd.concat(
    [
        assign_prps(pd.read_csv(path)).assign(Year=int(path.parent.name.split("_")[1]))
        for path in sorted(BENCHMARK_ROOT.glob("tpwrs_*/adjudicated_ra_affinities.csv"))
    ],
    ignore_index=True,
)
assignments_df["Affinity_Level"] = assignments_df["Affinity_Level"].astype(LEVEL_DTYPE)
assignments_df["Era"] = pd.cut(assignments_df["Year"], bins=ERA_BINS, labels=ERA_LABELS)

papers_per_year = assignments_df.groupby("Year")["Abstract_Index"].nunique().rename("Papers_Total")
papers_per_era = (
    assignments_df.groupby("Era", observed=True)["Abstract_Index"].nunique().rename("Papers_Total")
)
YEARS = papers_per_year.index.tolist()
YEAR_TICKTEXT = [
    f"{year}<br><span style='font-size:9px'>{papers_per_year[year]}"
    f"{'*' if year in PARTIAL_YEARS else ''}</span>"
    for year in YEARS
]


print(assignments_df.shape, "rows |", len(YEARS), "years |", papers_per_year.sum(), "papers")
papers_per_year
# papers_per_year.sum()

(36030, 8) rows | 12 years | 6005 papers


Year
2015    358
2016    531
2017    498
2018    703
2019    511
2020    483
2021    559
2022    451
2023    499
2024    602
2025    452
2026    358
Name: Papers_Total, dtype: int64

In [3]:
def style_year_axis(fig, dense=True, **selector):
    """Year ticks carrying each year's paper count, plus a shaded band over the partial year."""
    if dense:
        fig.update_xaxes(tickmode="array", tickvals=YEARS, ticktext=YEAR_TICKTEXT,
                         range=[YEARS[0] - 0.4, YEARS[-1] + 0.4], **selector)
    else:
        fig.update_xaxes(tickmode="array", tickvals=YEARS[::3], ticktext=[str(y) for y in YEARS[::3]],
                         range=[YEARS[0] - 0.2, YEARS[-1] + 0.2], **selector)

    # fig.add_vrect(x0=min(PARTIAL_YEARS) - 0.5, x1=YEARS[-1] + 0.3, fillcolor=PARTIAL_BAND,
    #               line_width=0, layer="below", **selector)

def add_footnote(fig, text=FOOTNOTE, y=-0.16):
    fig.add_annotation(x=0, y=y, xref="paper", yref="paper", text=text, showarrow=False,
                       xanchor="left", yanchor="top", font=dict(color=INK_MUTED, size=10))

def subtitle(text):
    return f"<br><span style='font-size:12px;color:{INK_SECONDARY}'>{text}</span>"


def share_by(*group_columns, totals):
    """Share of each group's papers that reach each threshold in THRESHOLDS."""
    frames = [
        assignments_df[assignments_df["Affinity_Level"].isin(levels)]
        .groupby(list(group_columns), observed=True)["Abstract_Index"].nunique()
        .rename("Papers_Reaching").reset_index().assign(Threshold=label)
        for label, levels in THRESHOLDS.items()
    ]
    share = pd.concat(frames, ignore_index=True).merge(totals.reset_index(), on=totals.index.name)
    share["Share"] = 100 * share["Papers_Reaching"] / share["Papers_Total"]
    return share

## 3. Headline: how much of TPWRS touches the agenda?

Two nested thresholds, both as a share of the year's papers. `Moderate+` is near-saturated context
(~80-90% of papers touch at least one programme at Moderate or better); `High` is the series that
actually moves, so it carries the accent and `Moderate+` recedes to grey.

A level-count line chart cannot answer this question: a paper holds one level *per programme*, so
"papers with a Low match" is ~99% of every year by construction and the three level lines sum to
roughly twice the paper count.

In [4]:
reach_share = share_by("Year", totals=papers_per_year)
all_year_reach = (
    reach_share.groupby("Threshold", as_index=False)[["Papers_Reaching", "Papers_Total"]]
    .sum()
    .assign(Year="all")
)
all_year_reach["Share"] = 100 * all_year_reach["Papers_Reaching"] / all_year_reach["Papers_Total"]


In [5]:

headline_fig = px.line(
    reach_share,
    x="Year",
    y="Share",
    color="Threshold",
    color_discrete_map={"Moderate+": CONTEXT, "High": ACCENT},
    markers=True,
    line_shape="linear",
    category_orders={"Threshold": list(THRESHOLDS.keys())},
    labels={"Share": "Share of papers (%)"},
)

for threshold, colour in [("Moderate+", CONTEXT), ("High", ACCENT)]:
    series = reach_share[reach_share["Threshold"] == threshold]
    last = series.iloc[-1]
    first = series.iloc[0]

    headline_fig.add_annotation(
        x=last["Year"] - 1.7,
        y=last["Share"] - 5.5,
        text=f"<b>{threshold}",
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(color=colour, size=12),
    )

    headline_fig.add_annotation(
            x=first["Year"],
            y=first["Share"]+13,
            text=f"2015-2026: <b>{all_year_reach.loc[all_year_reach['Threshold'] == threshold, 'Share'].values[0]:.0f}%</b> of all papers",
            showarrow=False,
            xanchor="left",
            yanchor="middle",
            font=dict(color=colour, size=12),
        )



headline_fig.update_layout(
    width=FIGURE_DIM[0],
    height=FIGURE_DIM[1],
    # margin_r=150,
    showlegend=False,
    title=dict(
        text="TPWRS papers touching the 2025 Research Agenda",
    ),
)

style_year_axis(headline_fig)
headline_fig.update_yaxes(title=None, ticksuffix="%", range=[0, 100], dtick=20)
headline_fig.update_xaxes(title=None)
add_footnote(headline_fig, text = "*2026 partial (collected to Aug 2026).")

# reach_share.pivot(index="Year", columns="Threshold", values="Share").round(1)
n = 4
headline_fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+60))
headline_fig.show()

In [6]:
paper_best_level = (
    assignments_df
    .sort_values(["Year", "Abstract_Index", "Highest_Question_Affinity"], ascending=[True, True, False])
    .drop_duplicates(["Year", "Abstract_Index"], keep="first")
    [["Year", "Era", "Abstract_Index", "Affinity_Level", "PRP_Name", "Highest_Question_Affinity"]]
    .rename(columns={"PRP_Name": "Top_PRP_Name"})
    .reset_index(drop=True)
)

relevance_mix = (
    paper_best_level.groupby(["Year", "Affinity_Level"], observed=True).size()
    .rename("Papers").reset_index()
)
relevance_mix["Share"] = (
    100 * relevance_mix["Papers"] / relevance_mix.groupby("Year")["Papers"].transform("sum")
)

mix_fig = px.bar(
    relevance_mix,
    x="Year",
    y="Share",
    color="Affinity_Level",
    color_discrete_map=LEVEL_COLORS,
)
for trace in mix_fig.data:
    trace.text = relevance_mix.loc[
        relevance_mix["Affinity_Level"] == trace.name, "Share"
    ].map(lambda value: f"{value:.1f}")
    trace.texttemplate = "%{text}"
    trace.textposition = "inside"
mix_fig.update_yaxes(title=None, ticksuffix="%", range=[0, 100], dtick=20)
mix_fig.show()


## 4. Programme coverage over time

One facet per programme, with the two thresholds drawn as **nested bands**: the dark band is High, the
band above it takes the stack up to Moderate+. High is anchored to the baseline, so its growth reads
against a flat reference instead of floating on top of a moving segment.

The y-axis is shared, which is what makes the facets comparable - CROF and Stability reach two to three
times as many papers as DER. A `Low` line is not plotted because it is exactly `100 - Moderate - High`.

In [7]:
coverage_share = share_by("Year", "PRP_Name", totals=papers_per_year)

coverage_bands = (
    coverage_share.pivot(index=["Year", "PRP_Name"], columns="Threshold", values="Share")
    .assign(Moderate=lambda frame: frame["Moderate+"] - frame["High"])
    .reset_index()
)
all_year_coverage = (
    coverage_share.groupby(["Threshold", "PRP_Name"], as_index=False)[["Papers_Reaching", "Papers_Total"]]
    .sum()
    .assign(Year="all")
)
all_year_coverage["Share"] = 100 * all_year_coverage["Papers_Reaching"] / all_year_coverage["Papers_Total"]

coverage_fig = px.area(coverage_bands,
                    x="Year", y=["High", "Moderate"], color_discrete_map={"High": LEVEL_COLORS["High"], "Moderate": LEVEL_COLORS["Moderate"]},
    facet_col="PRP_Name", facet_col_wrap=2, facet_row_spacing=0.04,
    # labels={"value": "Share of papers (%)", "variable": "Threshold"},
    category_orders={"PRP_Name": PROGRAMME_ORDER, "variable": ["High", "Moderate"]},
)
coverage_fig.update_layout(
    width=FIGURE_DIM[0],height=FIGURE_DIM[1]*2, title=dict(text="TPWRS share of papers per programe"),
    legend=dict(title_text="", orientation="h", yanchor="bottom", y=-0.1, xanchor="left", x=0),
)
coverage_fig.update_yaxes(title=None, range=[0, 80],ticksuffix="%", dtick=20)
coverage_fig.update_xaxes(title=None, dtick=3)

n = 4
coverage_fig.update_layout(margin=dict(l=n, r=n+30, t=n+50, b=n+60))
for annotation in coverage_fig.layout.annotations:
    annotation.text = annotation.text.split("=", 1)[-1]


coverage_totals = (
    all_year_coverage
    .pivot(index="PRP_Name", columns="Threshold", values="Share")
    # .reindex(PROGRAMME_ORDER)
)

for subplot_title in coverage_fig.layout.annotations:
    programme = subplot_title.text
    values = coverage_totals.loc[programme]

    coverage_fig.add_annotation(
        x=subplot_title.x - 0.05,
        y=subplot_title.y - 0.015,
        xref="paper",
        yref="paper",
        text=(
            f"2015–2026: <b>{values['Moderate+']:.1f}%</b> Moderate+"
            f"<br><b>{values['High']:.1f}%</b> High"
        ),
        showarrow=False,
        xanchor="center",
        yanchor="top",
        font=dict(color=INK_SECONDARY, size=10),
    )

coverage_fig.show()


## 5. Era shift, net of single-year noise

A single year holds 358-703 papers, so the 95% CI half-width on a share is about +/-1 pp where the share
is ~2% and +/-4 pp where it is ~60%. Several of the year-to-year wiggles above are inside that. Pooling
into four three-year eras (~1,400-1,700 papers each) and plotting only the endpoints shows what actually
moved.

Note that the era view corrects a misreading the annual endpoints invite: DER looks like it *fell*
between 2015 and 2026 (23.7% -> 19.0%), but pooled it is slightly up (18.5% -> 21.7%). Planning is the
one programme genuinely flat-to-down on both thresholds.

In [8]:
era_coverage = share_by("Era", "PRP_Name", totals=papers_per_era)
era_shift = (
    era_coverage[era_coverage["Era"].isin([ERA_LABELS[0], ERA_LABELS[-1]])]
    .pivot(index=["Threshold", "PRP_Name"], columns="Era", values="Share")
    .assign(Change=lambda frame: frame[ERA_LABELS[-1]] - frame[ERA_LABELS[0]])
    .reset_index()
)
aux = (
    era_shift
    .melt(
        id_vars=["Threshold", "PRP_Name", "Change"],
        var_name="Era",
        value_name="Share",
    )
)
aux["Change_direction"] = pd.Categorical(
    aux["Change"].lt(0).map({True: "+", False: "-"}),
    categories=["+", "-"],
    ordered=True,
)

In [9]:

coverage_era_shift = px.line(
    aux,
    x="Share",
    y="PRP_Name",
    symbol="PRP_Name",
    color="Change_direction",
    color_discrete_map={"-": LOSS, "+": GAIN},
    facet_col="Threshold",
    markers=True,
    category_orders={
        "Threshold": ["Moderate+", "High"],
        "PRP_Name": PROGRAMME_ORDER,
        # "Change_direction": ["+", "-"],
    },
)
coverage_era_shift.update_xaxes(matches=None, ticksuffix="%", title = None)
coverage_era_shift.update_yaxes(title = None)
coverage_era_shift.update_layout(
    width=FIGURE_DIM[0]*2,
    height=FIGURE_DIM[1],
    title=dict(text=f"Programme coverage shift, {ERA_LABELS[0]} \u2192 {ERA_LABELS[-1]}"),
    showlegend = False,
)
add_footnote(coverage_era_shift, text=f"Three-year pools (\u2248 1,400\u20131,700 papers each), so the move clears single-year sampling noise", y=-0.15)

n = 4
coverage_era_shift.update_layout(margin=dict(l=n, r=n+30, t=n+50, b=n+60))
for annotation in coverage_era_shift.layout.annotations:
    annotation.text = annotation.text.split("=", 1)[-1]

for col, threshold in enumerate(["Moderate+", "High"], start=1):
    for _, row in era_shift[era_shift["Threshold"] == threshold].iterrows():
        programme = row["PRP_Name"]
        change = row["Change"]

        coverage_era_shift.add_annotation(
            x=row[ERA_LABELS[-1]],
            y=programme,
            text=f"<b>{change:+.1f} pp</b>",
            xshift=20,
            xanchor="left",
            yanchor="middle",
            showarrow=False,
            font=dict(
                color=LOSS if change >= 0 else GAIN,
                size=10,
            ),
            row=1,
            col=col,
        )

coverage_era_shift.show()


In [10]:


# shift_fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.07,
#                           subplot_titles=("Moderate or better", "High only"))
# for annotation in shift_fig.layout.annotations:
#     annotation.update(font=dict(size=13, color=INK))

# for col, threshold in enumerate(["Moderate+", "High"], start=1):
#     panel = era_shift[era_shift["Threshold"] == threshold].set_index("PRP_Name").reindex(row_order)
#     limit = panel[[ERA_LABELS[0], ERA_LABELS[-1]]].to_numpy().max()
#     for programme, row in panel.iterrows():
#         start, end = row[ERA_LABELS[0]], row[ERA_LABELS[-1]]
#         colour = GAIN if end >= start else LOSS
#         shift_fig.add_trace(go.Scatter(
#             x=[start, end], y=[programme, programme], mode="lines",
#             line=dict(color=colour, width=6), showlegend=False, hoverinfo="skip",
#         ), row=1, col=col)
#         shift_fig.add_trace(go.Scatter(
#             x=[start, end], y=[programme, programme], mode="markers",
#             marker=dict(size=[11, 13], color=[SURFACE, colour], line=dict(color=colour, width=2.5)),
#             showlegend=False,
#             hovertemplate=f"<b>{programme}</b> {threshold}<br>"
#                           f"{ERA_LABELS[0]} {start:.1f}% \u2192 {ERA_LABELS[-1]} {end:.1f}%<extra></extra>",
#         ), row=1, col=col)
#         shift_fig.add_annotation(
#             x=limit * 1.08, y=programme, xanchor="left", yanchor="middle", showarrow=False,
#             text=f"<b>{row['Change']:+.1f}</b> pp",
#             font=dict(color=GAIN if row["Change"] >= 0 else LOSS, size=11), row=1, col=col,
#         )
#     shift_fig.update_xaxes(range=[0, limit * 1.34], ticksuffix="%", showgrid=True, gridcolor=GRID,
#                            row=1, col=col)

# shift_fig.update_layout(
#     height=430, width=1100, margin=dict(l=130, r=40, t=112, b=80),
#     title=dict(text=f"Programme coverage shift, {ERA_LABELS[0]} \u2192 {ERA_LABELS[-1]}"
#                     + subtitle("Three-year pools (\u2248 1,400\u20131,700 papers each), so the move "
#                                "clears single-year sampling noise")),
# )
# shift_fig.update_yaxes(showgrid=False, ticks="", tickfont=dict(color=INK_SECONDARY, size=12))
# add_footnote(shift_fig, text=f"Open dot {ERA_LABELS[0]}, filled dot {ERA_LABELS[-1]}. Sorted by the "
#                              "Moderate+ change, and the same order is kept in both panels.", y=-0.15)
# shift_fig.show()

# era_coverage.pivot(index="Era", columns=["Threshold", "PRP_Name"], values="Share")[
#     ["Moderate+", "High"]
# ].round(1)

## 6. Is the shift statistically real?

Section 5 compares two pooled endpoints, which on its own says nothing about whether a movement is
larger than sampling variation. The Cochran-Armitage trend test answers that: it treats every paper
as one Bernoulli trial and asks whether the reaching rate moves monotonically with the publication
year, weighting each year by the papers it actually holds. Regressing the twelve annual percentages
instead would treat the 358-paper 2015 and the 703-paper 2018 as equally informative.

`z` carries the sign of the trend and `p` is the two-sided normal tail. Since 2026 is partial, each
test is repeated without it, and `Robust` flags whether the significant/not verdict survives.

Every programme except DER and Planning rises significantly at both thresholds, and all of those
are robust. The two exceptions are the ones to read carefully, because both flip when the partial
year is dropped: DER's Moderate+ trend goes from n.s. to significant (z 1.78 -> 2.46) and Planning's
goes from n.s. to a significant *decline* (z -1.24 -> -2.16). The partial 2026 sample is small and
happens to sit low for both, so neither breadth trend should be stated more strongly than "flat,
with the direction sensitive to the final year". Planning's High trend is n.s. either way.

One further caveat: the test looks for a *linear* trend in the year scores, so a rise-then-fall
would partially cancel and report a small `z`. Read it next to the four era values in Section 5,
which show the shape.


In [11]:
from math import erfc, sqrt


def trend_test(panel):
    """Cochran-Armitage: does the reaching rate move monotonically with the publication year?"""
    panel = panel.sort_values("Year")
    years = panel["Year"].to_numpy(float)
    totals = panel["Papers_Total"].to_numpy(float)
    reaching = panel["Papers_Reaching"].to_numpy(float)

    pooled_rate = reaching.sum() / totals.sum()
    statistic = (years * (reaching - totals * pooled_rate)).sum()
    variance = pooled_rate * (1 - pooled_rate) * (
        (totals * years**2).sum() - (totals * years).sum() ** 2 / totals.sum()
    )
    z = statistic / sqrt(variance)
    return z, erfc(abs(z) / sqrt(2))  # two-sided normal tail


trend_rows = []
for threshold in THRESHOLDS:
    corpus_panel = reach_share[reach_share["Threshold"] == threshold]
    programme_panels = [
        (
            programme,
            coverage_share[
                (coverage_share["Threshold"] == threshold)
                & (coverage_share["PRP_Name"] == programme)
            ],
        )
        for programme in PROGRAMME_ORDER
    ]

    for scope, panel in [("All programmes", corpus_panel), *programme_panels]:
        z, p_value = trend_test(panel)
        z_excluding_partial, p_excluding_partial = trend_test(
            panel[~panel["Year"].isin(PARTIAL_YEARS)]
        )
        trend_rows.append({
            "Threshold": threshold,
            "Scope": scope,
            "Years": panel["Year"].nunique(),
            "z": z,
            "p": p_value,
            "z_excl_2026": z_excluding_partial,
            "p_excl_2026": p_excluding_partial,
        })

trend_stats = pd.DataFrame(trend_rows)
trend_stats["Verdict"] = (
    trend_stats["z"].gt(0).map({True: "rising", False: "falling"})
    .where(trend_stats["p"] < 0.05, "n.s.")
)
# A verdict that depends on the partial year is the one worth reading twice.
trend_stats["Robust"] = trend_stats["p"].lt(0.05) == trend_stats["p_excl_2026"].lt(0.05)

print(f"years per panel: {sorted(trend_stats['Years'].unique())} (all 12 = no year silently dropped)")
trend_stats.assign(
    p=trend_stats["p"].map("{:.1e}".format),
    p_excl_2026=trend_stats["p_excl_2026"].map("{:.1e}".format),
).round({"z": 2, "z_excl_2026": 2})


years per panel: [np.int64(12)] (all 12 = no year silently dropped)


,Threshold,Scope,Years,z,p,z_excl_2026,p_excl_2026,Verdict,Robust
0,Moderate+,All programmes,12,5.30,1.2e-07,5.22,1.8e-07,rising,True
1,Moderate+,IBR,12,7.92,2.4e-15,6.83,8.6e-12,rising,True
2,Moderate+,Stability,12,9.86,6.1e-23,8.16,3.3e-16,rising,True
3,Moderate+,DER,12,1.75,8.1e-02,2.44,1.5e-02,n.s.,False
4,Moderate+,CROF,12,6.95,3.5e-12,6.05,1.5e-09,rising,True
5,Moderate+,System Services,12,5.63,1.8e-08,4.93,8.3e-07,rising,True
6,Moderate+,Planning,12,-1.23,2.2e-01,-2.14,3.2e-02,n.s.,False
7,High,All programmes,12,8.78,1.6e-18,6.72,1.8e-11,rising,True
8,High,IBR,12,5.75,8.7e-09,4.47,8.0e-06,rising,True
9,High,Stability,12,9.32,1.1e-20,6.40,1.6e-10,rising,True


## 7. Which programme owns the engaged literature

Section 4 asks "how many papers does this programme reach", which double-counts papers relevant to
several programmes. This asks the complementary question: of the papers that engage the agenda at all,
which programme holds each one's single best-matching question? Columns sum to 100%.


In [12]:
top_prp_mix = (
    paper_best_level[paper_best_level["Affinity_Level"] != "Low"]
    .groupby(["Era", "Top_PRP_Name"], observed=True).size().rename("Papers").reset_index()
)
top_prp_mix["Share"] = (
    100 * top_prp_mix["Papers"] / top_prp_mix.groupby("Era", observed=True)["Papers"].transform("sum")
)
top_prp_matrix = (
    top_prp_mix.pivot(index="Top_PRP_Name", columns="Era", values="Share")
    .reindex(PROGRAMME_ORDER)[ERA_LABELS]
)

top_prp_fig = px.imshow(top_prp_matrix, color_continuous_scale=BLUE_RAMP, aspect="auto", zmin=0)
top_prp_fig.update_traces(
    xgap=3, ygap=3, texttemplate="%{z:.0f}%", textfont=dict(size=12),
    hovertemplate="<b>%{y}</b> %{x}<br>%{z:.1f}% of engaged papers<extra></extra>",
)
top_prp_fig.update_layout(
    height=430, width=900, margin=dict(l=140, r=40, t=96, b=64),
    title=dict(text="Which programme owns the engaged literature"
                    + subtitle("Share of each era's Moderate-or-better papers whose single "
                               "best-matching question sits in that programme")),
    coloraxis_colorbar=dict(title="", ticksuffix="%", thickness=10, len=0.55, outlinewidth=0),
)
top_prp_fig.update_xaxes(title=None, showline=False, ticks="", side="bottom",
                         tickfont=dict(color=INK_SECONDARY, size=12))
top_prp_fig.update_yaxes(title=None, showgrid=False, ticks="",
                         tickfont=dict(color=INK_SECONDARY, size=12))
add_footnote(top_prp_fig, text="Columns sum to 100%. Papers whose best question is Low are excluded.",
             y=-0.1)
top_prp_fig.show()

top_prp_matrix.round(1)

Era,2015-17,2018-20,2021-23,2024-26
Top_PRP_Name,,,,
IBR,9.0,8.1,8.9,8.6
Stability,20.9,22.9,24.4,27.2
DER,9.3,10.8,11.5,9.7
CROF,31.8,28.0,28.3,30.2
System Services,7.8,10.4,11.5,9.0
Planning,21.1,19.9,15.3,15.3


In [13]:
question_counts = (
    assignments_df[assignments_df["Affinity_Level"] != "Low"]
    .groupby(["Highest_Question_ID", "PRP_Name"], observed=True)
    .size()
    .rename("High_Papers")
    .reset_index()
)

question_profile = (
    question_counts
    .sort_values(["PRP_Name", "High_Papers", "Highest_Question_ID"], ascending=[True, False, True])
    .assign(
        High_Share=lambda frame: frame["High_Papers"] / frame.groupby("PRP_Name")["High_Papers"].transform("sum"),
        Question_Rank=lambda frame: frame.groupby("PRP_Name").cumcount() + 1,
    )
)
question_profile["Cumulative_High_Share"] = question_profile.groupby("PRP_Name")["High_Share"].cumsum()

question_concentration = (
    question_profile.groupby("PRP_Name", observed=True)
    .agg(
        High_Papers=("High_Papers", "sum"),
        Questions_With_High=("Highest_Question_ID", "size"),
        Top_Question=("Highest_Question_ID", "first"),
        Top_Question_Share=("High_Share", "first"),
        HHI=("High_Share", lambda values: values.pow(2).sum()),
    )
    .assign(Effective_Questions=lambda frame: 1 / frame["HHI"])
    .reindex(PROGRAMME_ORDER)
    .reset_index()
)

question_fig = px.bar(
    question_counts,
    x="Highest_Question_ID",
    y="High_Papers",
    color="PRP_Name",
    # barmode="group",
    category_orders={"PRP_Name": PROGRAMME_ORDER},
    labels={
        "Highest_Question_ID": "Highest-question ID",
        "High_Papers": "High-affinity papers",
        "PRP_Name": "PRP",
    },
    title="High-affinity papers by highest-matching question and PRP",
)
question_fig.update_xaxes(type="category", title=None)
question_fig.show()

concentration_fig = px.line(
    question_profile,
    x="Question_Rank",
    y="Cumulative_High_Share",
    color = "PRP_Name",
    # facet_col="PRP_Name",
    # facet_col_wrap=2,
    category_orders={"PRP_Name": PROGRAMME_ORDER},
    # color_discrete_sequence=[ACCENT],
    markers=True,
    labels={
        "Question_Rank": "Question rank by High-paper count",
        "Cumulative_High_Share": "Cumulative share of programme High papers",
    },
    title="How concentrated is each programme's High-affinity coverage?",
)
concentration_fig.update_yaxes(ticksuffix="%", tickformat=".0%", range=[0, 1], title=None)
concentration_fig.update_xaxes(dtick=1, title=None)
concentration_fig.update_layout(showlegend=True, height=FIGURE_DIM[1] * 2)
for annotation in concentration_fig.layout.annotations:
    annotation.text = annotation.text.split("=", 1)[-1]
concentration_fig.show()

question_concentration.round({"Top_Question_Share": 3, "HHI": 3, "Effective_Questions": 1})

,PRP_Name,High_Papers,Questions_With_High,Top_Question,Top_Question_Share,HHI,Effective_Questions
0,IBR,1473,6,4,0.559,0.404,2.5
1,Stability,2998,5,9,0.651,0.469,2.1
2,DER,1277,7,12,0.370,0.248,4.0
3,CROF,3642,9,21,0.261,0.188,5.3
4,System Services,1757,9,36,0.366,0.240,4.2
5,Planning,2156,12,41,0.346,0.182,5.5


## 8. Exports


In [14]:
figures = {
    "headline_agenda_reach": headline_fig,
    # "relevance_mix_by_year": mix_fig,
    "programme_coverage_by_year": coverage_fig,
    "programme_coverage_era_shift": coverage_era_shift,
    # "top_programme_mix_by_era": top_prp_fig,
}

for slug, figure in figures.items():
    figure.write_image(
        str(FIGURE_DIR / f"{slug}.pdf"),
        width=figure.layout.width,
        height=figure.layout.height,
    )

tables = {
    "headline_agenda_reach": reach_share,
    "programme_coverage_by_year": coverage_share,
    "programme_coverage_era_shift": era_shift,
}
for slug, table in tables.items():
    table.to_excel(OUTPUT_DIR / f"{slug}.xlsx", index=False)

print(f"{len(figures)} figures -> {FIGURE_DIR}")
print(f"{len(tables)} Excel tables -> {OUTPUT_DIR}")

3 figures -> /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/notebooks/outputs/tpwrs_prp_publications_by_year/figures
3 Excel tables -> /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/notebooks/outputs/tpwrs_prp_publications_by_year


In [15]:
# save_to = r'/mnt/c/Users/pgallego/OneDrive - Imperial College London/RA/Research agenda/G-PST shared folder/IEEE-GPST Article/Shared documents/Abstracts classified'
